In [ ]:
# ─────────────────────────────────────────────────────────────
# Step 0 · User settings
# ─────────────────────────────────────────────────────────────
# Gene symbol / short name — used for file/folder naming only
PROTEIN_NAME  = "AKT1"

PROTEIN_ALIASES = ["AKT1", "PKB", "protein kinase B alpha", "RAC-alpha serine/threonine-protein kinase"]

RESOLUTION_CUTOFF = 2.5   # Ångström — X-ray only
TOP_N             = 5     # Top N pockets per structure
LIGAND_CUTOFF     = 5.0   # Ångström — ligand-near-pocket threshold
MAX_STRUCTURES    = 200   # Safety cap

PINNED_SNAPSHOT = None

# ─────────────────────────────────────────────────────────────
# Step 1 · Install dependencies
# ─────────────────────────────────────────────────────────────
import os, re, json, math, time, shutil, subprocess, datetime
from pathlib import Path

import pandas as pd
import requests

FPOCKET_BIN = "/content/fpocket/bin/fpocket"

print("=" * 60)
print("  Installing fpocket")
print("=" * 60)

subprocess.run("apt-get -qq update", shell=True, check=True)
subprocess.run("apt-get -qq install -y git build-essential zip",
               shell=True, check=True)

if not Path(FPOCKET_BIN).exists():
    if Path("/content/fpocket").exists():
        shutil.rmtree("/content/fpocket")
    subprocess.run(
        "git clone --depth 1 https://github.com/Discngine/fpocket.git "
        "/content/fpocket",
        shell=True, check=True,
    )
    subprocess.run("make", shell=True, cwd="/content/fpocket", check=True)

print("fpocket ready.\n")

# ─────────────────────────────────────────────────────────────
# Step 2 · Fetch PDB IDs from RCSB
# ─────────────────────────────────────────────────────────────
SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
HEADERS    = {"Content-Type": "application/json",
              "Accept":       "application/json"}

def _post_query(payload, label="query"):
    """POST a query payload; return parsed JSON or None."""
    for attempt in range(1, 4):
        try:
            r = requests.post(
                SEARCH_URL,
                data=json.dumps(payload),
                headers=HEADERS,
                timeout=60,
            )
        except requests.exceptions.RequestException as exc:
            print(f"    [{label}] Network error (attempt {attempt}): {exc}")
            time.sleep(5 * attempt)
            continue

        if r.status_code == 204:
            # 204 = valid response, zero hits — not a server error
            return {"result_set": [], "total_count": 0}

        if r.status_code != 200:
            print(f"    [{label}] HTTP {r.status_code} (attempt {attempt}): "
                  f"{r.text[:200]}")
            time.sleep(5 * attempt)
            continue

        raw = r.text.strip()
        if not raw:
            print(f"    [{label}] Empty body (attempt {attempt}), retrying...")
            time.sleep(5 * attempt)
            continue

        try:
            return json.loads(raw)
        except json.JSONDecodeError as exc:
            print(f"    [{label}] JSON error (attempt {attempt}): {exc}")
            time.sleep(5 * attempt)

    return None   # all attempts exhausted


def _build_xray_res_filter(resolution_cutoff):
    """Two terminal nodes shared by every query."""
    return [
        {
            "type": "terminal", "service": "text",
            "parameters": {
                "attribute": "exptl.method",
                "operator":  "exact_match",
                "value":     "X-RAY DIFFRACTION",
            },
        },
        {
            "type": "terminal", "service": "text",
            "parameters": {
                "attribute": "rcsb_entry_info.resolution_combined",
                "operator":  "less_or_equal",
                "value":     resolution_cutoff,
            },
        },
    ]


def search_by_entity_description(alias, resolution_cutoff):
    """Search rcsb_polymer_entity.pdbx_description for one alias."""
    payload = {
        "query": {
            "type": "group",
            "logical_operator": "and",
            "nodes": [
                {
                    "type": "terminal", "service": "text",
                    "parameters": {
                        "attribute": "rcsb_polymer_entity.pdbx_description",
                        "operator":  "contains_phrase",
                        "value":     alias,
                    },
                },
                *_build_xray_res_filter(resolution_cutoff),
            ],
        },
        "return_type": "entry",
        "request_options": {"paginate": {"start": 0, "rows": 10000}},
    }
    data = _post_query(payload, label=f'entity_desc:"{alias}"')
    if data is None:
        return []
    ids = [e["identifier"] for e in data.get("result_set", [])]
    if ids:
        print(f'    entity_desc "{alias}" → {len(ids)} hits')
    return ids


def search_fulltext_fallback(alias, resolution_cutoff):
    """Full-text search across all indexed text fields."""
    payload = {
        "query": {
            "type": "group",
            "logical_operator": "and",
            "nodes": [
                {
                    "type": "terminal", "service": "full_text",
                    "parameters": {"value": alias},
                },
                *_build_xray_res_filter(resolution_cutoff),
            ],
        },
        "return_type": "entry",
        "request_options": {"paginate": {"start": 0, "rows": 10000}},
    }
    data = _post_query(payload, label=f'fulltext:"{alias}"')
    if data is None:
        return []
    ids = [e["identifier"] for e in data.get("result_set", [])]
    if ids:
        print(f'    full_text  "{alias}" → {len(ids)} hits')
    return ids


def rcsb_search(aliases, resolution_cutoff):
    """
    Try entity-description search for each alias.
    If all come back empty, retry with full-text search.
    Returns a deduplicated list of uppercase PDB IDs.
    """
    seen = set()
    result = []

    print("  Phase 1: entity description search")
    for alias in aliases:
        for pid in search_by_entity_description(alias, resolution_cutoff):
            if pid.upper() not in seen:
                seen.add(pid.upper())
                result.append(pid.upper())

    if result:
        return result

    print("  Phase 1 returned nothing.")
    print("  Phase 2: full-text fallback search")
    for alias in aliases:
        for pid in search_fulltext_fallback(alias, resolution_cutoff):
            if pid.upper() not in seen:
                seen.add(pid.upper())
                result.append(pid.upper())

    return result


snapshot_data = None
if PINNED_SNAPSHOT and Path(PINNED_SNAPSHOT).exists():
    snapshot_data = json.loads(Path(PINNED_SNAPSHOT).read_text())
    PDB_IDS = snapshot_data["pdb_ids"]
    print(f"Loaded pinned snapshot from {snapshot_data['date_queried']} "
          f"({len(PDB_IDS)} structures). Skipping live RCSB search.")
else:
    print(f"Querying RCSB for {PROTEIN_NAME} "
          f"(X-ray, resolution ≤ {RESOLUTION_CUTOFF} Å) ...")
    print(f"Aliases: {PROTEIN_ALIASES}\n")

    PDB_IDS = rcsb_search(PROTEIN_ALIASES, RESOLUTION_CUTOFF)
    PDB_IDS = PDB_IDS[:MAX_STRUCTURES]

    if not PDB_IDS:
        raise RuntimeError(
            f"No PDB structures found for any alias in {PROTEIN_ALIASES}.\n"
            "Suggestions:\n"
            "  • Check the protein name at https://www.rcsb.org (try a text search)\n"
            "  • Add more aliases to PROTEIN_ALIASES\n"
            "  • Relax RESOLUTION_CUTOFF (e.g. 3.0)\n"
            "  • Verify the protein has X-ray structures in the PDB"
        )

    print(f"\nTotal unique structures: {len(PDB_IDS)}")
    print("Sample IDs:", PDB_IDS[:10], ("..." if len(PDB_IDS) > 10 else ""))

# ─────────────────────────────────────────────────────────────
# Step 3 · Directory setup
# ─────────────────────────────────────────────────────────────
BASE        = Path(f"/content/{PROTEIN_NAME}_fpocket_project")
PDB_DIR     = BASE / "pdbs_original"
CLEAN_DIR   = BASE / "pdbs_clean"
RESULTS_DIR = BASE / "fpocket_results"
FINAL_DIR   = BASE / "final_outputs"

for d in [PDB_DIR, CLEAN_DIR, RESULTS_DIR, FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

snapshot_path = FINAL_DIR / f"{PROTEIN_NAME}_pdb_snapshot.json"
snapshot_path.write_text(json.dumps(
    snapshot_data if snapshot_data is not None else {
        "protein_name":      PROTEIN_NAME,
        "aliases":            PROTEIN_ALIASES,
        "resolution_cutoff":  RESOLUTION_CUTOFF,
        "date_queried":       datetime.date.today().isoformat(),
        "pdb_ids":            PDB_IDS,
    },
    indent=2,
))
print(f"  Snapshot saved: {snapshot_path}")

# ─────────────────────────────────────────────────────────────
# Step 4 · Download PDB files
# ─────────────────────────────────────────────────────────────
print("\nDownloading PDB files...")

def download_pdb(pdb_id, out_path, retries=3, timeout=30):
    """GET one PDB file with retry + backoff, same pattern as _post_query."""
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, timeout=timeout)
            if r.status_code == 200 and len(r.text) > 1000:
                out_path.write_text(r.text)
                return True
            print(f"    [{pdb_id}] HTTP {r.status_code} (attempt {attempt})")
        except requests.exceptions.RequestException as exc:
            print(f"    [{pdb_id}] Network error (attempt {attempt}): {exc}")
        if attempt < retries:
            time.sleep(5 * attempt)
    return False

failed_dl = []
for pdb_id in PDB_IDS:
    out = PDB_DIR / f"{pdb_id}.pdb"
    if out.exists() and out.stat().st_size > 1000:
        continue
    if download_pdb(pdb_id, out):
        print(f"  ↓ {pdb_id}")
    else:
        print(f"  ✗ {pdb_id} (failed after 3 attempts)")
        failed_dl.append(pdb_id)

PDB_IDS = [p for p in PDB_IDS if p not in failed_dl]
print(f"  {len(failed_dl)} failed  |  {len(PDB_IDS)} ready for processing")

# ─────────────────────────────────────────────────────────────
# Step 5 · Metadata from RCSB Data API
# ─────────────────────────────────────────────────────────────
print("\nCollecting metadata...")

IGNORE_LIGANDS = {
    "HOH","WAT","DOD","DMS","ACT","ACE",
    "NA","CL","K","MG","CA","ZN","MN","FE","CU","NI",
    "SO4","PO4","GOL","EDO","PEG","MPD","FMT","TRS",
    "IMD","EPE","MES","HEP",
}

def safe_get(url, retries=3, timeout=15):
    for _ in range(retries):
        try:
            r = requests.get(url, timeout=timeout)
            if r.status_code == 200:
                return r.json()
        except Exception:
            time.sleep(2)
    return None

def find_ph(obj, depth=0):
    if depth > 10:
        return []
    values = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k.lower() == "ph" and isinstance(v, (int, float)):
                values.append(v)
            values.extend(find_ph(v, depth + 1))
    elif isinstance(obj, list):
        for item in obj:
            values.extend(find_ph(item, depth + 1))
    return values

metadata_rows = []
for pdb_id in PDB_IDS:
    entry = safe_get(f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}")
    if entry is None:
        metadata_rows.append({
            "PDB_ID": pdb_id,
            "Structure_Title": "Not found",
            "Resolution_A": None, "pH": None,
            "Complexed": "NA", "Ligand_Inhibitor": "NA",
        })
        continue

    title      = entry.get("struct", {}).get("title", "NA")
    res_list   = entry.get("rcsb_entry_info", {}).get("resolution_combined")
    resolution = res_list[0] if res_list else None
    ph_vals    = find_ph(entry)
    ph         = ph_vals[0] if ph_vals else "NA"

    nonpoly_ids = (entry
                   .get("rcsb_entry_container_identifiers", {})
                   .get("non_polymer_entity_ids", []))
    ligands = []
    for eid in nonpoly_ids:
        lig = safe_get(
            f"https://data.rcsb.org/rest/v1/core/nonpolymer_entity/{pdb_id}/{eid}"
        )
        if lig is None:
            continue
        np = lig.get("pdbx_entity_nonpoly", {})
        cid, name = np.get("comp_id", "NA"), np.get("name", "NA")
        if cid not in IGNORE_LIGANDS:
            ligands.append(f"{cid} ({name})")

    metadata_rows.append({
        "PDB_ID":         pdb_id,
        "Structure_Title": title,
        "Resolution_A":   resolution,
        "pH":             ph,
        "Complexed":      "Yes" if ligands else "No",
        "Ligand_Inhibitor": "; ".join(ligands) or "No ligand/inhibitor",
    })

metadata_df = pd.DataFrame(metadata_rows)
metadata_df.to_csv(FINAL_DIR / f"{PROTEIN_NAME}_metadata.csv", index=False)
print(f"  Metadata: {len(metadata_df)} entries saved.")

# ─────────────────────────────────────────────────────────────
# Step 6 · Clean PDB files (ATOM only)
# ─────────────────────────────────────────────────────────────
print("\nCleaning PDB files...")

valid_pdbs = []
for pdb_id in PDB_IDS:
    infile  = PDB_DIR   / f"{pdb_id}.pdb"
    outfile = CLEAN_DIR / f"{pdb_id}_clean.pdb"
    if not infile.exists():
        continue
    atom_lines = [l for l in infile.read_text(errors="ignore")
                  .splitlines(keepends=True) if l.startswith("ATOM")]
    if len(atom_lines) < 10:
        print(f"  Skipped (too few ATOM lines): {pdb_id}")
        continue
    outfile.write_text("".join(atom_lines) + "END\n")
    valid_pdbs.append(pdb_id)

PDB_IDS = valid_pdbs
print(f"  {len(PDB_IDS)} structures passed cleaning.")

# ─────────────────────────────────────────────────────────────
# Step 7 · Run fpocket
# ─────────────────────────────────────────────────────────────
print("\nRunning fpocket...")

fpocket_failed = []
for pdb_id in PDB_IDS:
    clean_pdb = CLEAN_DIR   / f"{pdb_id}_clean.pdb"
    final_out = RESULTS_DIR / f"{pdb_id}_clean_out"
    if final_out.exists():
        print(f"  Already done, skipping: {pdb_id}")
        continue
    res = subprocess.run(
        [FPOCKET_BIN, "-f", str(clean_pdb)],
        capture_output=True, text=True,
    )
    generated = CLEAN_DIR / f"{pdb_id}_clean_out"
    if generated.exists():
        shutil.move(str(generated), str(final_out))
        print(f"  ✓ {pdb_id}")
    else:
        print(f"  ✗ fpocket produced no output for {pdb_id}")
        fpocket_failed.append(pdb_id)

PDB_IDS = [p for p in PDB_IDS if p not in fpocket_failed]
print(f"  fpocket done. {len(fpocket_failed)} structures skipped.")

# ─────────────────────────────────────────────────────────────
# Step 8 · Extract pocket scores
# ─────────────────────────────────────────────────────────────
print("\nExtracting pocket scores...")

def extract_float(block, label):
    m = re.search(
        rf"^\s*{re.escape(label)}\s*:\s*([-+]?\d*\.?\d+)",
        block, flags=re.MULTILINE,
    )
    return float(m.group(1)) if m else None

fpocket_rows = []
for pdb_id in PDB_IDS:
    info = RESULTS_DIR / f"{pdb_id}_clean_out" / f"{pdb_id}_clean_info.txt"
    if not info.exists():
        continue
    blocks = re.split(r"\nPocket\s+", info.read_text(errors="ignore"))
    count  = 0
    for blk in blocks:
        m = re.match(r"(\d+)", blk.strip())
        if not m:
            continue
        fpocket_rows.append({
            "PDB_ID":             pdb_id,
            "Pocket":             int(m.group(1)),
            "Pocket_Score":       extract_float(blk, "Score"),
            "Druggability_Score": extract_float(blk, "Druggability Score"),
            "Volume":             extract_float(blk, "Volume"),
            "Alpha_Spheres":      extract_float(blk, "Number of Alpha Spheres"),
            "Polarity_Score":     extract_float(blk, "Polarity Score"),
            "Mean_Local_Hydro":   extract_float(blk, "Mean local hydrophobicity"),
        })
        count += 1
        if count == TOP_N:
            break

fpocket_df = pd.DataFrame(fpocket_rows)
fpocket_df.to_csv(
    FINAL_DIR / f"{PROTEIN_NAME}_top{TOP_N}_fpocket_scores.csv", index=False
)
print(f"  Score table: {len(fpocket_df)} pocket entries.")

# ─────────────────────────────────────────────────────────────
# Step 9 · Pocket geometry + nearby-ligand detection
# ─────────────────────────────────────────────────────────────
print("\nAnalysing pocket geometry and nearby ligands...")

def parse_pdb_atoms(path, records=("ATOM", "HETATM")):
    atoms = []
    try:
        with open(path, errors="ignore") as f:
            for line in f:
                if not line.startswith(records):
                    continue
                try:
                    atoms.append({
                        "record":  line[0:6].strip(),
                        "atom":    line[12:16].strip(),
                        "resname": line[17:20].strip(),
                        "chain":   line[21].strip(),
                        "resseq":  line[22:26].strip(),
                        "x": float(line[30:38]),
                        "y": float(line[38:46]),
                        "z": float(line[46:54]),
                    })
                except (ValueError, IndexError):
                    pass
    except IOError:
        pass
    return atoms

def min_dist(point, coords):
    px, py, pz = point
    best = float("inf")
    for qx, qy, qz in coords:
        d = math.sqrt((px-qx)**2 + (py-qy)**2 + (pz-qz)**2)
        if d < best:
            best = d
    return best

location_rows = []
for pdb_id in PDB_IDS:
    orig_pdb   = PDB_DIR     / f"{pdb_id}.pdb"
    pocket_dir = RESULTS_DIR / f"{pdb_id}_clean_out" / "pockets"
    if not orig_pdb.exists() or not pocket_dir.exists():
        continue

    all_atoms   = parse_pdb_atoms(orig_pdb, ("ATOM", "HETATM"))
    lig_atoms   = [a for a in all_atoms
                   if a["record"] == "HETATM"
                   and a["resname"] not in IGNORE_LIGANDS]

    for pnum in range(1, TOP_N + 1):
        pfile = pocket_dir / f"pocket{pnum}_atm.pdb"
        if not pfile.exists():
            continue
        patoms = parse_pdb_atoms(pfile, ("ATOM", "HETATM"))
        if not patoms:
            continue

        coords = [(a["x"], a["y"], a["z"]) for a in patoms]
        n  = len(coords)
        cx = sum(c[0] for c in coords) / n
        cy = sum(c[1] for c in coords) / n
        cz = sum(c[2] for c in coords) / n

        residues = sorted({f'{a["chain"]}:{a["resname"]}{a["resseq"]}'
                           for a in patoms})

        nearby = {}
        for lig in lig_atoms:
            key  = f'{lig["resname"]}:{lig["chain"]}:{lig["resseq"]}'
            dist = min_dist((lig["x"], lig["y"], lig["z"]), coords)
            if dist <= LIGAND_CUTOFF:
                nearby[key] = min(nearby.get(key, dist), dist)

        lig_str = "; ".join(
            f"{k} ({v:.2f} Å)"
            for k, v in sorted(nearby.items(), key=lambda x: x[1])
        ) or "None within cutoff"

        location_rows.append({
            "PDB_ID":          pdb_id,
            "Pocket":          pnum,
            "Center_X":        round(cx, 3),
            "Center_Y":        round(cy, 3),
            "Center_Z":        round(cz, 3),
            "Num_Pocket_Atoms": n,
            "Pocket_Residues": ", ".join(residues),
            "Nearby_Ligands":  lig_str,
            "Ligand_Cutoff_A": LIGAND_CUTOFF,
        })

location_df = pd.DataFrame(location_rows)
location_df.to_csv(
    FINAL_DIR / f"{PROTEIN_NAME}_top{TOP_N}_pocket_locations.csv", index=False
)
print(f"  Location table: {len(location_df)} rows.")

# ─────────────────────────────────────────────────────────────
# Step 10 · Composite scoring & ranking.
# ─────────────────────────────────────────────────────────────
print("\nRanking pockets...")

df = (fpocket_df
      .merge(location_df, on=["PDB_ID", "Pocket"], how="left")
      .merge(metadata_df, on="PDB_ID",             how="left"))

df["Resolution_A"] = pd.to_numeric(df["Resolution_A"], errors="coerce")
df["Ligand_Bonus"] = df["Nearby_Ligands"].apply(
    lambda x: 0.0 if pd.isna(x) or "None within" in str(x) else 1.0
)

def minmax_norm(s):
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return s * 0.0
    return (s - lo) / (hi - lo)

for col in ["Pocket_Score", "Druggability_Score", "Volume"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

WEIGHTS = {
    "Druggability_Score": 0.40,
    "Pocket_Score":       0.30,
    "Volume":             0.10,
    "Resolution_A":       0.10,   # applied to 1/resolution: lower Å = better
    "Ligand_Bonus":       0.10,
}

df["_ds_n"]  = minmax_norm(df["Druggability_Score"])
df["_ps_n"]  = minmax_norm(df["Pocket_Score"])
df["_vol_n"] = minmax_norm(df["Volume"])
df["_res_n"] = minmax_norm(1.0 / df["Resolution_A"].replace(0, float("nan")))
df           = df.fillna({"_ps_n": 0, "_ds_n": 0, "_vol_n": 0, "_res_n": 0})

def composite_score(df, w):
    return (
        w["Druggability_Score"] * df["_ds_n"]  +
        w["Pocket_Score"]       * df["_ps_n"]  +
        w["Volume"]             * df["_vol_n"] +
        w["Resolution_A"]       * df["_res_n"] +
        w["Ligand_Bonus"]       * df["Ligand_Bonus"]
    )

df["Final_Score"] = composite_score(df, WEIGHTS)

def weight_sensitivity_check(df, base_weights, perturb=0.10):
    """
    Perturb each weight by +/-`perturb` (renormalized to sum to 1),
    recompute Final_Score, and check whether the top-ranked
    (PDB_ID, Pocket) changes. Prints each unstable perturbation and
    returns (is_stable, base_top_pick).
    """
    base_top  = df.sort_values("Final_Score", ascending=False).iloc[0]
    base_pick = (base_top["PDB_ID"], int(base_top["Pocket"]))
    stable    = True

    for key in base_weights:
        for direction in (1 + perturb, 1 - perturb):
            trial = dict(base_weights)
            trial[key] *= direction
            total = sum(trial.values())
            trial = {k: v / total for k, v in trial.items()}

            trial_score = composite_score(df, trial)
            top_row = (df.assign(_trial=trial_score)
                         .sort_values("_trial", ascending=False)
                         .iloc[0])
            pick = (top_row["PDB_ID"], int(top_row["Pocket"]))
            if pick != base_pick:
                stable = False
                print(f"    Unstable: {key} x{direction:.2f} "
                      f"shifts top pick to {pick[0]} pocket {pick[1]}")

    return stable, base_pick

print("\nRunning weight sensitivity check (+/-10% per weight)...")
is_stable, top_pick = weight_sensitivity_check(df, WEIGHTS)
if is_stable:
    print(f"  Stable: {top_pick[0]} pocket {top_pick[1]} stays top-ranked "
          f"under every +/-10% weight perturbation tested.")
else:
    print("  Not stable — see perturbations flagged above. The weighting "
          "scheme is worth revisiting before reporting this ranking.")

df.drop(columns=["_ps_n","_ds_n","_vol_n","_res_n"], inplace=True)

ranked_df = df.sort_values("Final_Score", ascending=False).reset_index(drop=True)
ranked_df.to_csv(
    FINAL_DIR / f"{PROTEIN_NAME}_top{TOP_N}_ranked_pockets.csv", index=False
)

best = ranked_df.iloc[0]
print("\n" + "=" * 60)
print("  BEST BINDING POCKET")
print("=" * 60)
print(f"  PDB ID             : {best['PDB_ID']}")
print(f"  Pocket #           : {int(best['Pocket'])}")
print(f"  Final Score        : {best['Final_Score']:.4f}")
print(f"  Pocket Score       : {best['Pocket_Score']}")
print(f"  Druggability Score : {best['Druggability_Score']}")
print(f"  Volume (Å³)        : {best['Volume']}")
print(f"  Resolution (Å)     : {best['Resolution_A']}")
print(f"  Nearby Ligands     : {best['Nearby_Ligands']}")
print(f"  Ligand/Inhibitor   : {best['Ligand_Inhibitor']}")
print(f"  Pocket Centre      : ({best['Center_X']}, {best['Center_Y']}, {best['Center_Z']})")
print(f"  Residues           : {best['Pocket_Residues']}")
print("=" * 60)

show_cols = ["PDB_ID","Pocket","Final_Score","Pocket_Score",
             "Druggability_Score","Volume","Resolution_A",
             "Nearby_Ligands","Ligand_Inhibitor"]
print("\nTop 10 ranked pockets:")
print(ranked_df[[c for c in show_cols if c in ranked_df.columns]]
      .head(10).to_string(index=False))

# ─────────────────────────────────────────────────────────────
# Step 11 · Zip & download
# ─────────────────────────────────────────────────────────────
print("\nCreating zip archive...")

zip_path = BASE / f"{PROTEIN_NAME}_fpocket_top{TOP_N}_results.zip"
if zip_path.exists():
    zip_path.unlink()

subprocess.run(f"zip -r {zip_path} {FINAL_DIR} {RESULTS_DIR}",
               shell=True, check=True)

print(f"\nOutput files in: {FINAL_DIR}")
for f in sorted(FINAL_DIR.iterdir()):
    print(f"  {f.name}")
print(f"\nZip: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("(Not in Colab — skipping auto-download.)")